# Simulador 17.2 — Optimización bajo Restricciones del Programa
### *Aprendizaje y Comportamiento Adaptable: Principios y Modelos*
**Capítulo 17: Optimización en Equilibrio**

---

Este simulador implementa el **modelo de distancia mínima de Staddon** — equivalente al modelo de maximización de utilidad cuadrática de Rachlin — y muestra cómo la geometría de la restricción del programa determina la forma de la función de respuesta en equilibrio.

### Sobre los parámetros

**Bliss point** $B_0 = (R_0, r_0)$: la distribución de comportamiento que el organismo ocuparía en ausencia de restricciones. Para un organismo típico de laboratorio, $R_0$ es pequeño (trabaja poco en libertad) y $r_0$ es moderado-alto (prefiere bastante consumo). Los valores por defecto ($R_0 = 3$, $r_0 = 8$) reflejan esa realidad.

**Peso b/a**: el costo de alejarse del consumo preferido relativo al costo de alejarse del trabajo preferido. Con $b/a = 2$, el consumo pesa el doble que el trabajo — consistente con que el organismo dedica más tiempo libre al consumo que al trabajo.

**Slider "Valor / Tasa ref."**:
- Para **RV**: es el número de respuestas requeridas por refuerzo (*n*). Mayor = más exigente.
- Para **IV**: es la tasa de refuerzo programada ($\lambda$, en ref/min). Mayor = programa más rico.

La función de retroalimentación IV tiene la forma $r = \lambda R \,/\, (R + K)$ con $K = 20$ fijo.

---


## Guía de exploración

**Básicas**
- Con programa RV, mueve el slider de izquierda a derecha (valor creciente). Sigue el punto naranja en el panel derecho. ¿En qué valor del programa se alcanza el máximo de $R^*$?
- Cambia el programa a IV. ¿Cómo cambia la forma de la función de respuesta? ¿Y la forma de la restricción en el panel izquierdo?

**Intermedias**
- Con RV, aumenta $b/a$ de 2.0 a 5.0. ¿Qué le ocurre al máximo de la función bitónica? Pista: el peso mayor en el consumo hace que el organismo trabaje más para compensar pérdidas de consumo.
- Mueve el bliss point a $R_0 = 8$, $r_0 = 4$. ¿Cómo cambia el comportamiento del modelo bajo RV? ¿La función sigue siendo bitónica? ¿Por qué?
- Con IV, reduce $r_0$ hasta que sea menor que el valor del slider ($\lambda$). ¿Qué ocurre con $R^*$? Interpreta.

**Avanzada**
- El capítulo dice que la igualación es el equilibrio de optimización bajo dos restricciones cóncavas simultáneas (programas concurrentes IV-IV). Usando el panel izquierdo, explica geométricamente por qué ninguna reasignación desde el punto óptimo puede aumentar la utilidad.
- ¿Qué ocurre con $R^*$ cuando el bliss point $B_0$ está *sobre* la curva de restricción? ¿Qué situación experimental representa ese caso?


In [ ]:
# ── Colab: ejecuta esta celda primero ─────────────────────────────────────────
# Si ves un error de widgets, descomenta las dos líneas siguientes:
# !pip install -q ipywidgets
# from google.colab import output; output.enable_custom_widget_manager()

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown, Layout
from scipy.optimize import minimize_scalar
import warnings
warnings.filterwarnings('ignore')

AZUL    = '#2C5282'
NARANJA = '#C05621'
VERDE   = '#276749'
GRIS    = '#718096'

try:
    matplotlib.rc('font', family='Georgia')
except Exception:
    pass

# Parámetro fijo de la forma de la restriccion IV
# r = lambda*R/(R + K_IV),  K_IV grande da curva suave (ganancias decrecientes)
K_IV = 20.0


def f_RV(R, n):
    """RV — restricción lineal: r = R/n.
    n = valor del programa (resp/ref). Mayor n = más exigente."""
    return R / n


def f_IV(R, lam):
    """IV — restricción cóncava: r = lam*R/(R + K_IV).
    lam = tasa de refuerzo programada (ref/min). Mayor lam = programa más rico."""
    return lam * R / (R + K_IV)


def get_fb(programa, n_val):
    if programa == 'RV':
        return lambda R: f_RV(R, n_val)
    return lambda R: f_IV(R, n_val)


def punto_optimo(R0, r0, a, b, programa, n_val):
    """Minimiza C(R) = a(R0-R)^2 + b(r0-f(R))^2 para encontrar R*."""
    fb = get_fb(programa, n_val)
    def costo(R):
        return a*(R0 - R)**2 + b*(r0 - fb(R))**2
    res = minimize_scalar(costo, bounds=(0.001, max(R0*5, 70.0)),
                          method='bounded')
    return res.x, fb(res.x)


def calc_funcion_respuesta(R0, r0, a, b, programa):
    if programa == 'RV':
        ns     = np.linspace(0.2, 35.0, 250)
        xlabel = ('Valor del programa RV  (resp/ref)\n'
                  'Mayor valor = programa más exigente')
        titulo = 'Función de respuesta:  RV\n(función bitónica)'
    else:
        ns     = np.linspace(0.5, 14.0, 250)
        xlabel = ('Tasa de refuerzo programada  (ref/min)\n'
                  'Mayor tasa = programa más rico')
        titulo = 'Función de respuesta:  IV\n(ganancias decrecientes)'
    R_stars = np.array([punto_optimo(R0, r0, a, b, programa, n)[0] for n in ns])
    return ns, R_stars, xlabel, titulo


def sim2(programa='RV', n_val=2.0, R0=3.0, r0=8.0, b_sobre_a=2.0):
    a, b = 1.0, b_sobre_a
    LMAX = 26

    Rg = np.linspace(0.02, LMAX, 380)
    rg = np.linspace(0.02, LMAX, 380)
    RR, rr = np.meshgrid(Rg, rg)
    U_q   = -(a*(RR - R0)**2 + b*(rr - r0)**2)
    U_min = np.nanmin(U_q)
    levels = np.linspace(U_min * 0.88, U_min * 0.04, 9)

    R_line = np.linspace(0, LMAX, 500)
    fb     = get_fb(programa, n_val)
    r_c    = np.clip(np.array([fb(R) for R in R_line]), 0, LMAX)

    R_star, r_star = punto_optimo(R0, r0, a, b, programa, n_val)
    ns_full, Rs_full, xlabel_r, titulo_r = calc_funcion_respuesta(R0, r0, a, b, programa)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15.5, 6.5))
    fig.patch.set_facecolor('white')

    # ── panel izquierdo ───────────────────────────────────────────────────
    ax1.set_facecolor('white')
    ax1.spines[['top', 'right']].set_visible(False)

    ax1.contour(RR, rr, U_q, levels=levels,
                colors=[AZUL]*len(levels), linewidths=1.4, alpha=0.55)

    ax1.plot(R0, r0, 'o', color=NARANJA, markersize=12, zorder=6,
             label=f'Bliss point $B_0=({R0:.0f},{r0:.0f})$')

    color_c = NARANJA if programa == 'RV' else VERDE
    lbl_c   = ('Restricción lineal (RV)' if programa == 'RV'
               else 'Restricción cóncava (IV)')
    ax1.plot(R_line, r_c, color=color_c, lw=2.5, label=lbl_c, zorder=5)

    color_star = VERDE if programa == 'RV' else NARANJA
    ax1.plot(R_star, r_star, '*', color=color_star, markersize=17, zorder=7,
             label=r'$R^*$=' + f'{R_star:.1f},  ' + r'$r^*$=' + f'{r_star:.2f}')

    ax1.plot([0, R_star], [r_star, r_star], '--', color=GRIS, lw=0.9, alpha=0.6)
    ax1.plot([R_star, R_star], [0, r_star], '--', color=GRIS, lw=0.9, alpha=0.6)

    c_opt = a*(R0 - R_star)**2 + b*(r0 - r_star)**2
    ax1.text(0.02, 0.97, f'Costo min.: {c_opt:.2f}',
             transform=ax1.transAxes, va='top', fontsize=10, color=GRIS,
             bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                       edgecolor=GRIS, alpha=0.9))

    tipo_txt = ('Restricción lineal\n(razón variable)' if programa == 'RV'
                else 'Restricción cóncava\n(intervalo variable)')
    ax1.text(0.98, 0.04, tipo_txt, transform=ax1.transAxes,
             ha='right', fontsize=9, color=color_c,
             bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                       edgecolor=color_c, alpha=0.9))

    ax1.set_xlim(0, LMAX); ax1.set_ylim(0, LMAX)
    ax1.set_xlabel('Tasa de trabajo  $R$  (resp/min)', fontsize=11)
    ax1.set_ylabel('Tasa de consumo  $r$  (ref/min)', fontsize=11)
    ax1.set_title(
        'Geometría: curvas de indiferencia y restriccion\n'
        '(modelo de distancia minima — Staddon)',
        fontsize=11, color=AZUL)
    ax1.grid(True, alpha=0.18, color=GRIS)
    ax1.legend(fontsize=9.5, loc='upper right')

    # ── panel derecho ─────────────────────────────────────────────────────
    ax2.set_facecolor('white')
    ax2.spines[['top', 'right']].set_visible(False)

    ax2.plot(ns_full, Rs_full, color=AZUL, lw=2.5,
             label='$R^*$ predicho (modelo)')
    ax2.axvline(n_val, color=GRIS, lw=1.0, ls='--', alpha=0.6)
    ax2.plot(n_val, R_star, 'o', color=color_star, markersize=12, zorder=5,
             label=f'Programa actual = {n_val:.1f}')

    ax2.set_title(titulo_r, fontsize=11, color=AZUL)
    ax2.set_xlabel(xlabel_r, fontsize=10)
    ax2.set_ylabel('Tasa de respuesta optima  $R^*$  (resp/min)', fontsize=11)
    ax2.grid(True, alpha=0.18, color=GRIS)
    ax2.legend(fontsize=10)

    suptitle = (f'Optimizacion en Equilibrio  |  Programa: {programa}'
                f'    B0=({R0:.0f},{r0:.0f})    b/a={b_sobre_a:.1f}')
    fig.suptitle(suptitle, fontsize=13, fontweight='bold', color=AZUL, y=1.01)
    plt.tight_layout()
    plt.show()


SL = dict(style={'description_width': '170px'},
          layout=Layout(width='480px'))

interact(sim2,
    programa  = Dropdown(
        options=['RV', 'IV'],
        description='Tipo de programa:',
        style={'description_width': '170px'},
        layout=Layout(width='330px')),
    n_val     = FloatSlider(
        min=0.2, max=35.0, step=0.2, value=2.0,
        description='Valor / Tasa ref.:',
        style={'description_width': '170px'},
        layout=Layout(width='480px')),
    R0        = FloatSlider(min=0.5, max=12.0, step=0.5, value=3.0,
                            description='R0 trabajo pref.:', **SL),
    r0        = FloatSlider(min=2.0, max=14.0, step=0.5, value=8.0,
                            description='r0 consumo pref.:', **SL),
    b_sobre_a = FloatSlider(min=0.25, max=6.0, step=0.25, value=2.0,
                            description='peso b/a:', **SL),
)
